# KuaiRand 用户行为分析

## 目标

在整体指标对比之后，继续从时间、用户、漏斗和内容维度拆解正常推荐与随机曝光的差异。

本部分重点回答：推荐带来的长播率提升，是否集中在特定时段、特定用户或特定内容类型。


## 1. 数据读取与清洗口径

比较同期随机曝光与正常推荐日志。沿用前文规则：删除完全重复记录，保留零视频时长记录，按小时使用 `hourmin // 100`。


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/raw/KuaiRand-Pure/data"

log_random = pd.read_csv(
    DATA_DIR / "log_random_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

log_standard = pd.read_csv(
    DATA_DIR / "log_standard_4_22_to_5_08_pure.csv"
).drop_duplicates().copy()

logs = {
    "随机曝光": log_random,
    "正常推荐": log_standard,
}

for name, log in logs.items():
    log["event_hour"] = log["hourmin"] // 100
    log["duration_missing"] = (log["duration_ms"] == 0).astype("int8")

    print("=" * 60)
    print(name)
    print("记录数：", f"{len(log):,}")
    print("日期范围：", log["date"].min(), "-", log["date"].max())
    print("小时范围：", log["event_hour"].min(), "-", log["event_hour"].max())


## 2. 小时维度指标计算

按小时聚合曝光量、活跃用户数、长播率、互动率、负反馈率和平均播放时长。


In [ ]:
def hourly_metrics(log: pd.DataFrame, group_name: str) -> pd.DataFrame:
    grouped = log.groupby("event_hour")
    result = pd.DataFrame({
        "exposure_count": grouped.size(),
        "active_users": grouped["user_id"].nunique(),
        "exposed_videos": grouped["video_id"].nunique(),
        "long_view_rate": grouped["long_view"].mean(),
        "click_rate": grouped["is_click"].mean(),
        "like_rate": grouped["is_like"].mean(),
        "comment_rate": grouped["is_comment"].mean(),
        "follow_rate": grouped["is_follow"].mean(),
        "forward_rate": grouped["is_forward"].mean(),
        "hate_rate": grouped["is_hate"].mean(),
        "avg_play_time_ms": grouped["play_time_ms"].mean(),
        "duration_missing_rate": grouped["duration_missing"].mean(),
    }).reset_index()

    result["group"] = group_name
    return result

hourly = pd.concat(
    [
        hourly_metrics(log_random, "随机曝光"),
        hourly_metrics(log_standard, "正常推荐"),
    ],
    ignore_index=True,
)


## 3. 24 小时趋势

按小时顺序展示完整趋势，避免只看 Top 排序造成误读。


In [ ]:
trend_columns = [
    "group",
    "event_hour",
    "exposure_count",
    "active_users",
    "long_view_rate",
    "click_rate",
    "like_rate",
    "hate_rate",
    "avg_play_time_ms",
]

hourly_trend = hourly.sort_values(["group", "event_hour"])[trend_columns]


In [ ]:
hourly_trend_display = hourly_trend.copy()

for column in ["long_view_rate", "click_rate", "like_rate", "hate_rate"]:
    hourly_trend_display[column] = hourly_trend_display[column].map(lambda x: f"{x:.2%}")

for column in ["exposure_count", "active_users"]:
    hourly_trend_display[column] = hourly_trend_display[column].map(lambda x: f"{int(x):,}")

hourly_trend_display["avg_play_time_ms"] = hourly_trend_display["avg_play_time_ms"].map(lambda x: f"{x:,.0f}")

hourly_trend_display


## 4. 小时差异对比

按小时对齐两组数据，计算正常推荐相对随机曝光的长播率差异。


In [ ]:
def pivot_hourly_metric(metric: str) -> pd.DataFrame:
    pivot = hourly.pivot(
        index="event_hour",
        columns="group",
        values=metric,
    )
    pivot["正常推荐-随机曝光"] = pivot["正常推荐"] - pivot["随机曝光"]
    return pivot.reset_index()

long_view_by_hour = pivot_hourly_metric("long_view_rate")
click_by_hour = pivot_hourly_metric("click_rate")
hate_by_hour = pivot_hourly_metric("hate_rate")


In [ ]:
long_view_by_hour_display = long_view_by_hour.rename(columns={
    "event_hour": "小时",
    "正常推荐": "正常推荐长播率",
    "随机曝光": "随机曝光长播率",
    "正常推荐-随机曝光": "长播率差异",
}).copy()

for column in ["正常推荐长播率", "随机曝光长播率"]:
    long_view_by_hour_display[column] = long_view_by_hour_display[column].map(lambda x: f"{x:.2%}")

long_view_by_hour_display["长播率差异"] = long_view_by_hour_display["长播率差异"].map(
    lambda x: f"{x * 100:+.2f} 个百分点"
)

long_view_by_hour_display


In [ ]:
summary_by_hour = long_view_by_hour.merge(
    hate_by_hour[["event_hour", "正常推荐-随机曝光"]].rename(
        columns={"正常推荐-随机曝光": "负反馈率差异"}
    ),
    on="event_hour",
)
summary_by_hour = summary_by_hour.rename(
    columns={"正常推荐-随机曝光": "长播率差异"}
)

summary_by_hour["长播率差异_百分点"] = summary_by_hour["长播率差异"] * 100
summary_by_hour["负反馈率差异_百分点"] = summary_by_hour["负反馈率差异"] * 100

hour_gap_display = (
    summary_by_hour
    .sort_values("长播率差异", ascending=False)
    .head(10)
    [["event_hour", "长播率差异_百分点", "负反馈率差异_百分点"]]
    .rename(columns={
        "event_hour": "小时",
        "长播率差异_百分点": "长播率差异",
        "负反馈率差异_百分点": "负反馈率差异",
    })
    .copy()
)

for column in ["长播率差异", "负反馈率差异"]:
    hour_gap_display[column] = hour_gap_display[column].map(lambda x: f"{x:+.2f} 个百分点")

hour_gap_display


## 5. Top 时段校验

Top 小时需同时查看曝光量和活跃用户数，避免小样本比例波动。


In [ ]:
top_hours = (
    hourly
    .sort_values(["group", "long_view_rate"], ascending=[True, False])
    .groupby("group")
    .head(5)
    [[
        "group",
        "event_hour",
        "exposure_count",
        "active_users",
        "long_view_rate",
        "avg_play_time_ms",
        "hate_rate",
    ]]
)

top_hours_display = top_hours.copy()
top_hours_display["exposure_count"] = top_hours_display["exposure_count"].map(lambda x: f"{int(x):,}")
top_hours_display["active_users"] = top_hours_display["active_users"].map(lambda x: f"{int(x):,}")
top_hours_display["long_view_rate"] = top_hours_display["long_view_rate"].map(lambda x: f"{x:.2%}")
top_hours_display["avg_play_time_ms"] = top_hours_display["avg_play_time_ms"].map(lambda x: f"{x:,.0f}")
top_hours_display["hate_rate"] = top_hours_display["hate_rate"].map(lambda x: f"{x:.2%}")

top_hours_display


## 6. 时间维度结论

正常推荐在主要时段的长播率均高于随机曝光，推荐效果在小时维度上较稳定。

Top 时段不能单独解释为用户天然更活跃，仍需结合曝光量、视频类型、视频时长和用户结构进一步拆解。


## 7. 用户活跃度维度

按 `user_active_degree` 拆解，观察正常推荐优势是否集中在某类用户中。


In [ ]:
users = pd.read_csv(DATA_DIR / "user_features_pure.csv")

user_active_counts = (
    users["user_active_degree"]
    .value_counts(dropna=False)
    .rename_axis("user_active_degree")
    .reset_index(name="用户数")
)

user_active_counts["用户数"] = user_active_counts["用户数"].map(lambda x: f"{int(x):,}")
user_active_counts


样本量较小的类别不单独做强解释，后续归入“其他”。


In [ ]:
def map_active_group(value: str) -> str:
    if value in ["full_active", "high_active"]:
        return "高活跃"
    if value == "middle_active":
        return "中活跃"
    if value in ["low_active", "single_low_active"]:
        return "低活跃"
    if value in ["day_new", "2_14_day_new"]:
        return "新用户"
    return "其他"

users["active_group"] = users["user_active_degree"].map(map_active_group)

active_group_counts = (
    users["active_group"]
    .value_counts()
    .rename_axis("活跃度分组")
    .reset_index(name="用户数")
)
active_group_counts["用户数"] = active_group_counts["用户数"].map(lambda x: f"{int(x):,}")
active_group_counts


In [ ]:
logs_with_user = {}

for name, log in logs.items():
    merged = log.merge(
        users[["user_id", "user_active_degree", "active_group"]],
        on="user_id",
        how="left",
        validate="many_to_one",
    )
    logs_with_user[name] = merged

    print("=" * 60)
    print(name)
    print("合并前：", f"{len(log):,}")
    print("合并后：", f"{len(merged):,}")
    print("活跃度缺失：", merged["active_group"].isna().sum())


In [ ]:
def user_group_metrics(log: pd.DataFrame, group_name: str) -> pd.DataFrame:
    grouped = log.groupby("active_group")
    result = pd.DataFrame({
        "exposure_count": grouped.size(),
        "active_users": grouped["user_id"].nunique(),
        "long_view_rate": grouped["long_view"].mean(),
        "click_rate": grouped["is_click"].mean(),
        "like_rate": grouped["is_like"].mean(),
        "hate_rate": grouped["is_hate"].mean(),
        "avg_play_time_sec": grouped["play_time_ms"].mean() / 1000,
    }).reset_index()
    result["group"] = group_name
    return result

user_group_result = pd.concat(
    [
        user_group_metrics(logs_with_user["随机曝光"], "随机曝光"),
        user_group_metrics(logs_with_user["正常推荐"], "正常推荐"),
    ],
    ignore_index=True,
)


In [ ]:
user_group_display = user_group_result.copy()

for column in ["long_view_rate", "click_rate", "like_rate", "hate_rate"]:
    user_group_display[column] = user_group_display[column].map(lambda x: f"{x:.2%}")

for column in ["exposure_count", "active_users"]:
    user_group_display[column] = user_group_display[column].map(lambda x: f"{int(x):,}")

user_group_display["avg_play_time_sec"] = user_group_display["avg_play_time_sec"].map(lambda x: f"{x:.2f}")

user_group_display = user_group_display.rename(columns={
    "active_group": "用户活跃度分组",
    "group": "曝光机制",
    "exposure_count": "曝光量",
    "active_users": "活跃用户数",
    "long_view_rate": "长播率",
    "click_rate": "有效播放反馈率",
    "like_rate": "点赞率",
    "hate_rate": "负反馈率",
    "avg_play_time_sec": "平均播放时长_秒",
})

user_group_display


## 8. 用户维度结论

正常推荐在各活跃度分组中的长播率均高于随机曝光，说明推荐优势并非只由某一类用户贡献。

各活跃度组的提升幅度整体较稳定，用户活跃度暂不是解释推荐效果差异的主要因素。后续应继续从视频类型、视频时长和内容标签等内容维度寻找差异来源。


## 9. 漏斗维度

从曝光到长播、互动和负反馈，观察推荐提升主要出现在哪些行为环节。


In [ ]:
def funnel_metrics(log, group_name):
    exposure = len(log)

    result = pd.DataFrame({
        "环节": [
            "曝光",
            "有效播放反馈",
            "长播",
            "点赞",
            "评论",
            "关注",
            "负反馈"
        ],
        "人数/次数": [
            exposure,
            log["is_click"].sum(),
            log["long_view"].sum(),
            log["is_like"].sum(),
            log["is_comment"].sum(),
            log["is_follow"].sum(),
            log["is_hate"].sum()
        ]
    })

    result["曝光机制"] = group_name
    result["占曝光比例"] = result["人数/次数"] / exposure

    return result

funnel = pd.concat(
    [
        funnel_metrics(log_random, "随机曝光"),
        funnel_metrics(log_standard, "正常推荐")
    ],
    ignore_index=True
)


In [ ]:
funnel_display = funnel.copy()

funnel_display["人数/次数"] = funnel_display["人数/次数"].map(
    lambda x: f"{int(x):,}"
)

funnel_display["占曝光比例"] = funnel_display["占曝光比例"].map(
    lambda x: f"{x:.2%}"
)

funnel_display

## 10. 漏斗维度结论

正常推荐在有效播放反馈、长播和正向互动环节均高于随机曝光，说明推荐提升不是只发生在单一行为上。

负反馈率没有同步放大，说明推荐在提升观看深度的同时，没有明显增加用户反感行为。


## 11. 内容维度：视频时长

视频时长会影响长播率，因此需要检查推荐优势是否只是由时长结构造成。


In [ ]:
video_basic = pd.read_csv(DATA_DIR / "video_features_basic_pure.csv")

logs_with_video = {}

for name, log in logs.items():
    merged = log.merge(
        video_basic[["video_id", "video_duration", "tag"]],
        on="video_id",
        how="left",
        validate="many_to_one",
    )
    logs_with_video[name] = merged

    print("=" * 60)
    print(name)
    print("合并前：", f"{len(log):,}")
    print("合并后：", f"{len(merged):,}")
    print("video_duration 缺失：", merged["video_duration"].isna().sum())
    print("tag 缺失：", merged["tag"].isna().sum())


In [ ]:
def duration_bucket(duration_ms):
    if pd.isna(duration_ms) or duration_ms <= 0:
        return "未知"
    duration_sec = duration_ms / 1000
    if duration_sec <= 10:
        return "0-10秒"
    if duration_sec <= 30:
        return "10-30秒"
    if duration_sec <= 60:
        return "30-60秒"
    return "60秒以上"

for log in logs_with_video.values():
    log["duration_bucket"] = log["video_duration"].map(duration_bucket)
    log["tag_clean"] = log["tag"].fillna("未知").astype(str)


In [ ]:
def content_group_metrics(log: pd.DataFrame, group_name: str, column: str) -> pd.DataFrame:
    grouped = log.groupby(column)
    result = pd.DataFrame({
        "exposure_count": grouped.size(),
        "active_users": grouped["user_id"].nunique(),
        "video_count": grouped["video_id"].nunique(),
        "long_view_rate": grouped["long_view"].mean(),
        "click_rate": grouped["is_click"].mean(),
        "like_rate": grouped["is_like"].mean(),
        "hate_rate": grouped["is_hate"].mean(),
        "avg_play_time_sec": grouped["play_time_ms"].mean() / 1000,
    }).reset_index()
    result["group"] = group_name
    return result

def format_metric_table(df: pd.DataFrame) -> pd.DataFrame:
    display = df.copy()
    for column in ["long_view_rate", "click_rate", "like_rate", "hate_rate"]:
        if column in display.columns:
            display[column] = display[column].map(lambda x: f"{x:.2%}")
    for column in ["exposure_count", "active_users", "video_count"]:
        if column in display.columns:
            display[column] = display[column].map(lambda x: f"{int(x):,}")
    if "avg_play_time_sec" in display.columns:
        display["avg_play_time_sec"] = display["avg_play_time_sec"].map(lambda x: f"{x:.2f}")
    return display


In [ ]:
duration_result = pd.concat(
    [
        content_group_metrics(logs_with_video["随机曝光"], "随机曝光", "duration_bucket"),
        content_group_metrics(logs_with_video["正常推荐"], "正常推荐", "duration_bucket"),
    ],
    ignore_index=True,
)

duration_display = format_metric_table(duration_result).rename(columns={
    "duration_bucket": "视频时长分组",
    "group": "曝光机制",
    "exposure_count": "曝光量",
    "active_users": "活跃用户数",
    "video_count": "视频数",
    "long_view_rate": "长播率",
    "click_rate": "有效播放反馈率",
    "like_rate": "点赞率",
    "hate_rate": "负反馈率",
    "avg_play_time_sec": "平均播放时长_秒",
})

duration_display


In [ ]:
duration_compare = (
    duration_result
    .pivot(index="duration_bucket", columns="group", values="long_view_rate")
    .reset_index()
)
duration_compare["长播率差异"] = duration_compare["正常推荐"] - duration_compare["随机曝光"]

duration_compare_display = duration_compare.rename(columns={
    "duration_bucket": "视频时长分组",
    "正常推荐": "正常推荐长播率",
    "随机曝光": "随机曝光长播率",
}).copy()

for column in ["正常推荐长播率", "随机曝光长播率"]:
    duration_compare_display[column] = duration_compare_display[column].map(lambda x: f"{x:.2%}")

duration_compare_display["长播率差异"] = duration_compare_display["长播率差异"].map(
    lambda x: f"{x * 100:+.2f} 个百分点"
)

duration_compare_display


## 12. 视频时长结论

按视频时长分层后，正常推荐在各时长段的长播率、有效播放反馈率和平均播放时长均高于随机曝光。

因此，整体长播率提升并不只是由视频时长结构造成；在同一时长区间内，推荐机制仍然能提升观看深度。

不同时长区间的差异没有 tag 维度明显，视频时长不是本项目中解释推荐效果差异的主要因素。


## 13. 内容维度：tag

`tag` 是匿名内容标签编码，本文只把它作为内容类型分层变量，不解释具体内容含义。

分析时先过滤低曝光标签，避免小样本比例波动。


In [ ]:
tag_min_exposure = 3000

tag_result = pd.concat(
    [
        content_group_metrics(logs_with_video["随机曝光"], "随机曝光", "tag_clean"),
        content_group_metrics(logs_with_video["正常推荐"], "正常推荐", "tag_clean"),
    ],
    ignore_index=True,
)

valid_tags = (
    tag_result
    .groupby("tag_clean")["exposure_count"]
    .sum()
    .loc[lambda s: s >= tag_min_exposure]
    .index
)

tag_result_filtered = tag_result[tag_result["tag_clean"].isin(valid_tags)].copy()


In [ ]:
tag_compare = (
    tag_result_filtered
    .pivot(index="tag_clean", columns="group", values="long_view_rate")
    .dropna()
    .reset_index()
)
tag_compare["长播率差异"] = tag_compare["正常推荐"] - tag_compare["随机曝光"]

tag_compare_display = (
    tag_compare
    .sort_values("正常推荐", ascending=False)
    .head(20)
    .rename(columns={
        "tag_clean": "tag",
        "正常推荐": "正常推荐长播率",
        "随机曝光": "随机曝光长播率",
    })
    .copy()
)

for column in ["正常推荐长播率", "随机曝光长播率"]:
    tag_compare_display[column] = tag_compare_display[column].map(lambda x: f"{x:.2%}")

tag_compare_display["长播率差异"] = tag_compare_display["长播率差异"].map(
    lambda x: f"{x * 100:+.2f} 个百分点"
)

tag_compare_display


In [ ]:
tag_lift_display = (
    tag_compare
    .sort_values("长播率差异", ascending=False)
    .head(20)
    .rename(columns={
        "tag_clean": "tag",
        "正常推荐": "正常推荐长播率",
        "随机曝光": "随机曝光长播率",
    })
    .copy()
)

for column in ["正常推荐长播率", "随机曝光长播率"]:
    tag_lift_display[column] = tag_lift_display[column].map(lambda x: f"{x:.2%}")

tag_lift_display["长播率差异"] = tag_lift_display["长播率差异"].map(
    lambda x: f"{x * 100:+.2f} 个百分点"
)

tag_lift_display


## 14. tag 维度结论

不同 tag 下推荐带来的长播率提升不一致，说明推荐效果与内容类型有关。

长播率差异较大的 tag，可能更依赖人群匹配：随机曝光时分发给大量非目标用户，正常推荐后更容易触达感兴趣用户。

随机曝光长播率较高且差异较小的 tag，可能更接近大众型内容，适合更广泛曝光。

由于 tag 是匿名编码，本文不解释具体内容类别，只保留“内容类型对推荐匹配敏感度不同”这一分析结论。


## 15. 本部分结论

正常推荐相对随机曝光的优势在小时、用户活跃度、漏斗和内容维度上均存在，整体表现较稳定。

用户活跃度和视频时长没有解释出主要差异；tag 维度的差异更明显，说明推荐提升更可能来自内容兴趣匹配和人群匹配。

后续 04 将进一步做归因分析，区分：整体提升来自内容结构变化，还是来自同一内容分组内的推荐效果提升。
